# Gold — Geography Dimension (Kimball)

Single-row geography dimension for Iceland.
Included per Kimball conventions — even a single-valued dimension should be modelled explicitly rather than hard-coded, as scope almost always expands.

**Output:** `gold.dim_geography` (Delta table)  
**Grain:** One row per country (1 row — Iceland)  
**Type:** SCD Type 0 — geography attributes are fixed  

| Column | Type | Description |
|---|---|---|
| `geography_key` | int | Surrogate key |
| `country_iso_code` | string | ISO 3166-1 alpha-2 country code |
| `country_name` | string | English country name |
| `region` | string | Geographic region |
| `currency_code` | string | ISO 4217 currency code |
| `currency_name` | string | Full currency name |

In [ ]:
import pandas as pd

geographies = [
    (1, "IS", "Iceland", "Northern Europe", "ISK", "Icelandic Króna"),
]

df_geo = pd.DataFrame(geographies, columns=[
    "geography_key", "country_iso_code", "country_name",
    "region", "currency_code", "currency_name"
])

print(df_geo.to_string(index=False))

In [ ]:
df_spark = spark.createDataFrame(df_geo)
df_spark.createOrReplaceTempView("v_dim_geography")

if not spark.catalog.tableExists("gold.dim_geography"):
    df_spark.write.format("delta").saveAsTable("gold.dim_geography")
else:
    spark.sql("""
        MERGE INTO gold.dim_geography AS target
        USING v_dim_geography AS source
        ON target.geography_key = source.geography_key
        WHEN MATCHED THEN UPDATE SET
            target.country_iso_code = source.country_iso_code,
            target.country_name     = source.country_name,
            target.region           = source.region,
            target.currency_code    = source.currency_code,
            target.currency_name    = source.currency_name
        WHEN NOT MATCHED THEN INSERT (
            geography_key, country_iso_code, country_name,
            region, currency_code, currency_name
        )
        VALUES (
            source.geography_key, source.country_iso_code, source.country_name,
            source.region, source.currency_code, source.currency_name
        )
    """)

print(f"Saved to gold.dim_geography - {spark.table('gold.dim_geography').count()} rows")